# AeroPulse PM2.5 — Phase 7: Tuning, Attribution and the Acceptance Gate

Tune for stability rather than for a lucky season, explain the model where it matters,
compare everything against an honest baseline, and then run the gate that decides whether
any of it may be promoted.

## Objective

Three jobs, in order.

**1. Tune for stability.** Optuna searches over `learning_rate`, `num_leaves`,
`max_depth`, `min_child_samples`, `feature_fraction`, `bagging_fraction`, `lambda_l1` and
`lambda_l2`, with **purged rolling-origin folds as the CV object** — never random K-fold.

The objective minimised is `mean(RMSE) + std(RMSE)` across folds, not `mean(RMSE)`. That
is deliberate: with one winter and one post-monsoon cycle in the data, a configuration can
win on the mean by fitting whichever season dominates the folds. Adding the standard
deviation makes the search prefer a configuration that is consistently decent over one
that is occasionally excellent, which is the correct preference for a deployed forecaster.

Hyperparameters are also **constrained not to worsen extreme bias** against the untuned
model. Tuning controls capacity and regularisation; the tail behaviour was set in notebook
06 by the loss and the sample weights, and a trial that trades tail accuracy for RMSE is
rejected rather than ranked.

**2. Attribute.** SHAP and permutation importance, broken out by season and by extreme
events. Global importance on a dataset that is 40% monsoon rows describes monsoon air; the
question worth asking is what drives the model *during an episode*.

**3. Gate.** The roadmap §23 acceptance criteria, checked and reported. Per `AGENTS.md`
the gate is load-bearing: **a model that misses a threshold stays at `VALIDATION` and the
threshold is not relaxed.** This notebook is allowed to fail its own gate, and if it does,
that is the finding.

| Gate | Threshold |
|---|---|
| Rolling-CV mean R² | ≥ 0.75 |
| Extreme recall | ≥ 0.70 |
| Extreme precision | ≥ 0.25 |
| Median event lead time | ≥ 24 h |

In [1]:
# ============================================================================
# SETUP
# ============================================================================
from pathlib import Path
import os, json, time, gc, warnings
import numpy as np
import pandas as pd
import joblib
from dotenv import find_dotenv, load_dotenv

import optuna
from lightgbm import LGBMRegressor
import phase7_eval as ev

try:
    from catboost import CatBoostRegressor
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False
try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False

_dotenv = find_dotenv(usecwd=True)
if _dotenv:
    load_dotenv(_dotenv, override=True)
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

PROJECT_ROOT = Path(os.getenv("AEROPULSE_PROJECT_ROOT", ".")).resolve()
EVENT_DIR = PROJECT_ROOT / "data" / "pm25" / "processed" / "event_aware"
EVENT_FILE = EVENT_DIR / "pm25_event_aware_features.parquet"
PHASE7 = PROJECT_ROOT / "artifacts" / "pm25" / "phase7"
OUT_DIR = PHASE7 / "tuning"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PRIMARY_H = int(os.getenv("PM25_PRIMARY_H", "24"))
# Must match notebook 06. At 3 folds the test blocks begin in mid-February and
# contain no winter or post-monsoon, so every extreme metric would be measured
# on seasons that have almost no extremes. 5 folds reaches the burning season.
N_SPLITS = int(os.getenv("PM25_CV_SPLITS", "5"))
N_TRIALS = int(os.getenv("PM25_OPTUNA_TRIALS", "20"))
EMBARGO_H = 48
# n_estimators is held FIXED during the search so trials compare regularisation
# at equal capacity rather than letting a trial win by simply growing more
# trees. The winner is refitted at the full count afterwards.
SEARCH_TREES = 300
FINAL_TREES = 400

ACCEPTANCE_GATES = {
    "rolling_cv_mean_r2": 0.75,
    "extreme_recall": 0.70,
    "extreme_precision": 0.25,
    "median_lead_time_h": 24.0,
}

manifest = json.loads((EVENT_DIR / "dataset_manifest.json").read_text())
FEATURES = manifest["features"]
GROUPS = manifest["feature_groups"]

EXP_REPORT = PHASE7 / "experiments" / "experiment_report.json"
HAZ_REPORT = PHASE7 / "peak_hazard" / "peak_hazard_report.json"
exp = json.loads(EXP_REPORT.read_text()) if EXP_REPORT.exists() else None
haz = json.loads(HAZ_REPORT.read_text()) if HAZ_REPORT.exists() else None
if exp is None:
    raise FileNotFoundError("Run notebook 06 first - the loss configuration is "
                            "selected there, not re-decided here.")

CHOSEN = exp["best_on_primary_target"]["experiment"]
# Read as data, not parsed from the label - see notebook 07's note. "E7_h24" is
# a tail-weighted run whose name contains no evidence of that fact.
_cfg = exp["best_on_primary_target"].get("loss_config")
if _cfg:
    USE_LOG = bool(_cfg["log_target"])
    USE_WEIGHTS = bool(_cfg["tail_weights"])
    QUANTILE = _cfg.get("quantile")
else:
    raise KeyError(
        "notebook 06's report has no loss_config block. Re-run notebook 06: "
        "tuning against a configuration guessed from a label would tune the "
        "wrong model.")

print(f"dataset v{manifest['dataset_version']} ({manifest['sha256_16']}), "
      f"{len(FEATURES)} features")
print(f"loss config from notebook 06: {CHOSEN} "
      f"(log1p={USE_LOG}, tail_weights={USE_WEIGHTS}, quantile={QUANTILE})")
print(f"optuna trials {N_TRIALS}, folds {N_SPLITS}, horizon t+{PRIMARY_H}h")
print(f"catboost={HAS_CATBOOST}  shap={HAS_SHAP}")

/Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


dataset v3.0.0 (ba72d91a76c075ee), 184 features
loss config from notebook 06: E2_tail_weights (log1p=False, tail_weights=True, quantile=None)
optuna trials 20, folds 5, horizon t+24h
catboost=True  shap=True


In [2]:
# ============================================================================
# LOAD AND FOLDS - the same purged rolling-origin splits as notebooks 06 and 07
# ============================================================================
TARGET = f"target_pm25_t_plus_{PRIMARY_H}h"
META = ["timestamp_utc", "location_id", "season", "pollution_regime"]

t0 = time.time()
df = pd.read_parquet(EVENT_FILE, columns=FEATURES + [TARGET] + META)
X = df[FEATURES].to_numpy("float32")
y = df[TARGET].to_numpy("float32")
TS = pd.DatetimeIndex(df["timestamp_utc"])
CUR = df["pm25"].to_numpy("float64")
meta = df[META].reset_index(drop=True)
del df
gc.collect()

ok = np.flatnonzero(~np.isnan(y))
FOLDS = [(ok[tr], ok[te], cut, end) for tr, te, cut, end in
         ev.rolling_origin_folds(TS[ok], n_splits=N_SPLITS,
                                 purge_hours=PRIMARY_H, embargo_hours=EMBARGO_H)]
print(f"Loaded {X.shape} in {time.time()-t0:.0f}s; {len(FOLDS)} folds")
fold_seasons = []
for i, (tr, te, cut, end) in enumerate(FOLDS, 1):
    s = meta.loc[te, "season"].mode().iloc[0]
    fold_seasons.append(s)
    print(f"  fold {i}: train {len(tr):>9,}  test {len(te):>8,}  "
          f"{pd.Timestamp(cut).date()} -> {pd.Timestamp(end).date()}  "
          f"{s:<13} extreme test rows {int((y[te] >= 150).sum()):,}")

# Same guard as notebook 06, for the same reason: tuning against folds that
# exclude the episode seasons optimises for clean air and then reports the
# result as if it were a fix for the extreme regime.
if not (set(fold_seasons) & {"post_monsoon", "winter"}):
    raise RuntimeError(
        "No test fold lands in post-monsoon or winter, so the tuning objective "
        "and the acceptance gate would both be measured on seasons with almost "
        f"no extremes. Raise PM25_CV_SPLITS (currently {N_SPLITS}).")


def fit_and_score(params, folds=None, trees=SEARCH_TREES, cols=None):
    """Fit LightGBM with the notebook-06 loss config; return per-fold metrics."""
    cols = cols or FEATURES
    ci = np.array([FEATURES.index(c) for c in cols])
    p = {"n_estimators": trees, "random_state": 42, "verbose": -1, "n_jobs": -1, **params}
    if QUANTILE is not None:
        p.update(objective="quantile", alpha=QUANTILE)
    rows, preds = [], {}
    for i, (tr, te, cut, end) in enumerate(folds or FOLDS, 1):
        y_tr, y_te = y[tr].astype("float64"), y[te].astype("float64")
        fit_y = np.log1p(np.clip(y_tr, 0, None)) if USE_LOG else y_tr
        w = ev.tail_weights(y_tr) if USE_WEIGHTS else None
        m = LGBMRegressor(**p).fit(X[np.ix_(tr, ci)], fit_y, sample_weight=w)
        raw = m.predict(X[np.ix_(te, ci)])
        pred = np.clip(np.expm1(raw) if USE_LOG else raw, 0, None)
        base = ev.persistence(CUR[te])
        rm = ev.regression_metrics(y_te, pred)
        em = ev.event_metrics(y_te, pred)
        rows.append({"fold": i, **rm,
                     **{k: v for k, v in em.items() if k != "threshold"},
                     "persistence_RMSE": ev.regression_metrics(y_te, base)["RMSE"],
                     "persistence_extreme_recall":
                         ev.event_metrics(y_te, base)["extreme_recall"]})
        preds[i] = (te, pred)
        del m
        gc.collect()
    return pd.DataFrame(rows), preds


print("\nUntuned reference (the notebook-06 configuration at search capacity):")
BASE_PARAMS = dict(learning_rate=0.05, num_leaves=63, min_child_samples=40,
                   subsample=0.9, subsample_freq=1, colsample_bytree=0.8)
base_folds, _ = fit_and_score(BASE_PARAMS)
BASELINE_OBJ = float(base_folds.RMSE.mean() + base_folds.RMSE.std())
BASELINE_XBIAS = float(base_folds.extreme_bias.mean())
print(base_folds.round(4).to_string(index=False))
print(f"\n  objective  mean+std RMSE = {BASELINE_OBJ:.4f}")
print(f"  extreme bias              = {BASELINE_XBIAS:+.2f} ug/m3   <- trials may "
      f"not make this worse")

Loaded (1705252, 184) in 4s; 5 folds
  fold 1: train   625,627  test  194,597  2025-10-03 -> 2025-12-10  post_monsoon  extreme test rows 6,018
  fold 2: train   819,354  test  160,736  2025-12-10 -> 2026-02-16  winter        extreme test rows 5,770
  fold 3: train   983,771  test  187,438  2026-02-16 -> 2026-04-25  pre_monsoon   extreme test rows 1,316
  fold 4: train 1,168,025  test  176,624  2026-04-25 -> 2026-07-02  pre_monsoon   extreme test rows 610
  fold 5: train 1,345,168  test  180,709  2026-07-02 -> 2026-09-08  monsoon       extreme test rows 382

Untuned reference (the notebook-06 configuration at search capacity):


 fold     MAE    RMSE      R2    bias  extreme_precision  extreme_recall  extreme_f1  extreme_bias  actual_events  predicted_events  persistence_RMSE  persistence_extreme_recall
    1 20.7422 35.2620  0.4030  6.2416             0.5234          0.2996      0.3811      -95.2423           6018              3445           37.7276                      0.4975
    2 21.0628 32.4757  0.4405 10.3157             0.4739          0.5721      0.5184      -46.0450           5770              6966           34.7847                      0.4872
    3 15.6201 24.2016  0.2707  8.0418             0.3137          0.1375      0.1912     -105.3989           1316               577           27.0976                      0.2090
    4 13.5571 26.7640 -0.0081  8.0028             0.0631          0.0230      0.0337     -219.7005            610               222           31.8644                      0.0623
    5  9.4495 18.7062  0.0631  4.6413             0.1269          0.0654      0.0864     -203.4958            

## 1. Optuna over purged rolling-origin folds

Each trial fits `N_SPLITS` models, so the search is deliberately small. That is the right
trade here: with a single winter in the data, the marginal value of trial 200 over trial 20
is far smaller than the risk of tuning into this particular year's seasonal accident.

In [3]:
# ============================================================================
# OPTUNA - CV object is the purged rolling-origin split, never random K-fold
# ============================================================================
def objective(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 255, log=True),
        "max_depth": trial.suggest_int("max_depth", 4, 14),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 300, log=True),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "subsample_freq": 1,
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    }
    folds, _ = fit_and_score(params)
    mean_rmse, std_rmse = float(folds.RMSE.mean()), float(folds.RMSE.std())
    xbias = float(folds.extreme_bias.mean())
    trial.set_user_attr("mean_rmse", mean_rmse)
    trial.set_user_attr("std_rmse", std_rmse)
    trial.set_user_attr("extreme_bias", xbias)
    trial.set_user_attr("extreme_recall", float(folds.extreme_recall.mean()))
    trial.set_user_attr("mean_r2", float(folds.R2.mean()))
    trial.set_user_attr("std_r2", float(folds.R2.std()))
    # Pessimistic RMSE: prefers a configuration that is consistently decent over
    # one that is occasionally excellent.
    return mean_rmse + std_rmse


t0 = time.time()
study = optuna.create_study(direction="minimize",
                            sampler=optuna.samplers.TPESampler(seed=42),
                            study_name="pm25_phase7_stability")
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
print(f"{len(study.trials)} trials in {time.time()-t0:.0f}s")

trials = pd.DataFrame([{
    "trial": t.number, "objective": t.value,
    **{k: t.user_attrs[k] for k in ["mean_rmse", "std_rmse", "mean_r2", "std_r2",
                                    "extreme_bias", "extreme_recall"]},
    **t.params,
} for t in study.trials if t.value is not None]).sort_values("objective")

# The constraint: tuning may not buy RMSE with tail accuracy. A trial whose
# extreme bias is further from zero than the untuned model's is rejected, not
# ranked - which is the whole point of having the constraint rather than a
# weighted objective that could silently trade the tail away.
trials["worsens_extreme_bias"] = trials.extreme_bias.abs() > abs(BASELINE_XBIAS)
eligible = trials[~trials.worsens_extreme_bias]
print(f"\n{len(eligible)}/{len(trials)} trials did not worsen extreme bias.")
display(trials.head(10)[["trial", "objective", "mean_rmse", "std_rmse", "mean_r2",
                         "extreme_bias", "extreme_recall",
                         "worsens_extreme_bias"]].round(4))

if eligible.empty:
    print("\nNo trial satisfied the extreme-bias constraint. Keeping the UNTUNED "
          "configuration - relaxing the constraint to produce a 'tuned' model would "
          "be exactly the threshold-shopping AGENTS.md forbids.")
    BEST_PARAMS, tuned_from = dict(BASE_PARAMS), "untuned (no eligible trial)"
else:
    row = eligible.iloc[0]
    BEST_PARAMS = {k: row[k] for k in
                   ["learning_rate", "num_leaves", "max_depth", "min_child_samples",
                    "colsample_bytree", "subsample", "reg_alpha", "reg_lambda"]}
    BEST_PARAMS["num_leaves"] = int(BEST_PARAMS["num_leaves"])
    BEST_PARAMS["max_depth"] = int(BEST_PARAMS["max_depth"])
    BEST_PARAMS["min_child_samples"] = int(BEST_PARAMS["min_child_samples"])
    BEST_PARAMS["subsample_freq"] = 1
    tuned_from = f"optuna trial {int(row.trial)}"
    print(f"\nSelected {tuned_from}:")
    for k, v in BEST_PARAMS.items():
        print(f"  {k:20} {v}")
    print(f"  objective {row.objective:.4f} vs untuned {BASELINE_OBJ:.4f} "
          f"({row.objective - BASELINE_OBJ:+.4f})")

20 trials in 1689s

19/20 trials did not worsen extreme bias.


,trial,objective,mean_rmse,std_rmse,mean_r2,extreme_bias,extreme_recall,worsens_extreme_bias
18,18,33.0254,27.3870,5.6384,0.2196,-132.7837,0.2362,False
19,19,33.0814,27.3219,5.7595,0.2266,-131.9892,0.2442,False
17,17,33.1240,27.3052,5.8188,0.2287,-131.3738,0.2485,False
13,13,33.1562,27.3872,5.7690,0.2227,-131.2544,0.2526,False
10,10,33.1919,27.3209,5.8709,0.2294,-131.2714,0.2539,False
5,5,33.2079,27.4074,5.8005,0.2224,-130.7084,0.2593,False
7,7,33.2988,27.4539,5.8449,0.2207,-131.8658,0.2432,False
15,15,33.3028,27.3758,5.9271,0.2265,-129.9658,0.2558,False
9,9,33.3246,27.4611,5.8634,0.2191,-132.8161,0.2246,False
14,14,33.3370,27.4343,5.9026,0.2229,-129.2128,0.2695,False



Selected optuna trial 18:
  learning_rate        0.01090088376259108
  num_leaves           20
  max_depth            7
  min_child_samples    172
  colsample_bytree     0.616899010333987
  subsample            0.6357468352541736
  reg_alpha            0.1611260657123359
  reg_lambda           0.680596446122901
  subsample_freq       1
  objective 33.0254 vs untuned 34.0727 (-1.0473)


### Did tuning actually help?

The comparison that matters is not "best trial vs first trial" — Optuna guarantees that
improves. It is **tuned vs untuned at equal capacity, on the same folds**, and whether the
difference exceeds the fold-to-fold spread. A gain smaller than the standard deviation
across folds is not a gain.

In [4]:
# ============================================================================
# TUNED vs UNTUNED at full capacity
# ============================================================================
tuned_folds, tuned_preds = fit_and_score(BEST_PARAMS, trees=FINAL_TREES)
untuned_folds, untuned_preds = fit_and_score(BASE_PARAMS, trees=FINAL_TREES)

cmp_rows = []
for label, f in [("untuned", untuned_folds), ("tuned", tuned_folds)]:
    cmp_rows.append({
        "config": label,
        "mean_R2": f.R2.mean(), "std_R2": f.R2.std(),
        "mean_RMSE": f.RMSE.mean(), "std_RMSE": f.RMSE.std(),
        "mean_MAE": f.MAE.mean(),
        "extreme_bias": f.extreme_bias.mean(),
        "extreme_recall": f.extreme_recall.mean(),
        "extreme_precision": f.extreme_precision.mean(),
        "persistence_RMSE": f.persistence_RMSE.mean(),
    })
tuning_effect = pd.DataFrame(cmp_rows)
tuning_effect["skill_vs_persistence"] = (
    1 - tuning_effect.mean_RMSE / tuning_effect.persistence_RMSE)
display(tuning_effect.round(4))

d_rmse = tuning_effect.mean_RMSE.iloc[0] - tuning_effect.mean_RMSE.iloc[1]
spread = tuning_effect.std_RMSE.mean()
print(f"Tuning changed mean RMSE by {d_rmse:+.3f} ug/m3, against a fold-to-fold "
      f"spread of {spread:.3f}.")
if abs(d_rmse) < spread:
    print("The gain is SMALLER than the fold-to-fold spread, so it is not "
          "distinguishable from seasonal composition. Reported as such rather than "
          "as an improvement.")
else:
    print("The gain exceeds the fold-to-fold spread, so it is a real effect on "
          "this data.")
print("\nPer-fold detail (tuned):")
print(tuned_folds.round(4).to_string(index=False))

,config,mean_R2,std_R2,mean_RMSE,std_RMSE,mean_MAE,extreme_bias,extreme_recall,extreme_precision,persistence_RMSE,skill_vs_persistence
0,untuned,0.2346,0.1973,27.4984,6.6860,16.0650,-135.3360,0.2067,0.2963,30.5073,0.0986
1,tuned,0.2261,0.2310,27.3392,5.7942,16.4342,-130.6239,0.2540,0.3307,30.5073,0.1038


Tuning changed mean RMSE by +0.159 ug/m3, against a fold-to-fold spread of 6.240.
The gain is SMALLER than the fold-to-fold spread, so it is not distinguishable from seasonal composition. Reported as such rather than as an improvement.

Per-fold detail (tuned):
 fold     MAE    RMSE      R2    bias  extreme_precision  extreme_recall  extreme_f1  extreme_bias  actual_events  predicted_events  persistence_RMSE  persistence_extreme_recall
    1 20.1118 34.1287  0.4408  6.8057             0.5820          0.4164      0.4855      -87.2432           6018              4306           37.7276                      0.4975
    2 20.5776 31.3904  0.4773  9.9271             0.5228          0.5201      0.5215      -49.5356           5770              5740           34.7847                      0.4872
    3 16.8820 25.1011  0.2155 10.2442             0.2765          0.1467      0.1917     -104.0666           1316               698           27.0976                      0.2090
    4 14.2254 26.8959 -0.0

## 2. Final model comparison

Every candidate on the same folds, against the persistence baseline that `AGENTS.md`
requires. CatBoost is included because it was the Phase-5 runner-up and a like-for-like
comparison is the only way to know whether LightGBM is the right family here.

In [5]:
# ============================================================================
# FINAL COMPARISON - persistence, untuned LGBM, tuned LGBM, CatBoost
# ============================================================================
final_rows = []

# Persistence, scored on exactly the fold test blocks the models saw.
pers = []
for i, (tr, te, cut, end) in enumerate(FOLDS, 1):
    y_te = y[te].astype("float64")
    p = ev.persistence(CUR[te])
    pers.append({"fold": i, **ev.regression_metrics(y_te, p),
                 **{k: v for k, v in ev.event_metrics(y_te, p).items()
                    if k != "threshold"}})
pers = pd.DataFrame(pers)


def summarise(label, folds, notes=""):
    return {"model": label,
            "mean_R2": folds.R2.mean(), "std_R2": folds.R2.std(),
            "mean_MAE": folds.MAE.mean(), "mean_RMSE": folds.RMSE.mean(),
            "std_RMSE": folds.RMSE.std(),
            "extreme_bias": folds.extreme_bias.mean(),
            "extreme_recall": folds.extreme_recall.mean(),
            "extreme_precision": folds.extreme_precision.mean(),
            "notes": notes}


final_rows.append(summarise("persistence", pers, "baseline required by AGENTS.md"))
final_rows.append(summarise("lightgbm_untuned", untuned_folds,
                            f"notebook-06 config ({CHOSEN})"))
final_rows.append(summarise("lightgbm_tuned", tuned_folds, tuned_from))

if HAS_CATBOOST:
    cat_rows = []
    for i, (tr, te, cut, end) in enumerate(FOLDS, 1):
        y_tr, y_te = y[tr].astype("float64"), y[te].astype("float64")
        fit_y = np.log1p(np.clip(y_tr, 0, None)) if USE_LOG else y_tr
        w = ev.tail_weights(y_tr) if USE_WEIGHTS else None
        cb = CatBoostRegressor(iterations=FINAL_TREES, learning_rate=0.05, depth=8,
                               random_seed=42, verbose=0, allow_writing_files=False)
        cb.fit(X[tr], fit_y, sample_weight=w)
        raw = cb.predict(X[te])
        pred = np.clip(np.expm1(raw) if USE_LOG else raw, 0, None)
        cat_rows.append({"fold": i, **ev.regression_metrics(y_te, pred),
                         **{k: v for k, v in ev.event_metrics(y_te, pred).items()
                            if k != "threshold"}})
        del cb
        gc.collect()
    cat_folds = pd.DataFrame(cat_rows)
    final_rows.append(summarise("catboost", cat_folds, "Phase-5 runner-up, same loss"))
else:
    cat_folds = pd.DataFrame()

final = pd.DataFrame(final_rows)
final["abs_extreme_bias"] = final.extreme_bias.abs()
final = final.sort_values(["abs_extreme_bias", "extreme_recall", "mean_RMSE"],
                          ascending=[True, False, True])
print("Ranked by |extreme_bias|, then extreme recall, then RMSE:")
display(final.drop(columns=["abs_extreme_bias"]).round(4).reset_index(drop=True))

pers_row = final[final.model == "persistence"].iloc[0]
# Same rule as notebook 06: ranking orders the table, but only a model that beats
# the baseline on RMSE may be called the best or carried into the gate. Ranking
# on extreme bias alone would crown whichever model hedges highest.
candidates = final[final.model != "persistence"]
models_only = candidates[candidates.mean_RMSE < pers_row.mean_RMSE]
dropped = candidates[candidates.mean_RMSE >= pers_row.mean_RMSE]
if len(dropped):
    print("Excluded for not beating persistence on RMSE: "
          f"{', '.join(dropped.model)}")
if models_only.empty:
    raise RuntimeError(
        "No model beats persistence on RMSE, so none may be promoted "
        "(AGENTS.md). The hazard classifier in notebook 07 carries the alerting "
        "claim and is evaluated on its own terms.")
print(f"Persistence : extreme recall {pers_row.extreme_recall:.3f}, "
      f"bias {pers_row.extreme_bias:+.1f}, RMSE {pers_row.mean_RMSE:.2f}")
print(f"Best model  : {models_only.iloc[0].model} -> recall "
      f"{models_only.iloc[0].extreme_recall:.3f}, bias "
      f"{models_only.iloc[0].extreme_bias:+.1f}, RMSE "
      f"{models_only.iloc[0].mean_RMSE:.2f}")
if models_only.extreme_recall.max() <= pers_row.extreme_recall:
    print("\nNO MODEL BEATS PERSISTENCE ON EXTREME RECALL. Per AGENTS.md a model "
          "that cannot beat its baseline must not be promoted on this metric - the "
          "calibrated hazard classifier from notebook 07 is the component that "
          "carries the alerting claim, and it is evaluated on its own terms below.")

Ranked by |extreme_bias|, then extreme recall, then RMSE:


,model,mean_R2,std_R2,mean_MAE,mean_RMSE,std_RMSE,extreme_bias,extreme_recall,extreme_precision,notes
0,persistence,0.0288,0.3358,14.4440,30.5073,6.5785,-124.6250,0.2910,0.2896,baseline required by AGENTS.md
1,lightgbm_tuned,0.2261,0.2310,16.4342,27.3392,5.7942,-130.6239,0.2540,0.3307,optuna trial 18
2,catboost,0.2362,0.2280,16.0368,27.2305,6.0816,-132.6918,0.2296,0.3416,"Phase-5 runner-up, same loss"
3,lightgbm_untuned,0.2346,0.1973,16.0650,27.4984,6.6860,-135.3360,0.2067,0.2963,notebook-06 config (E2_tail_weights)


Persistence : extreme recall 0.291, bias -124.6, RMSE 30.51
Best model  : lightgbm_tuned -> recall 0.254, bias -130.6, RMSE 27.34

NO MODEL BEATS PERSISTENCE ON EXTREME RECALL. Per AGENTS.md a model that cannot beat its baseline must not be promoted on this metric - the calibrated hazard classifier from notebook 07 is the component that carries the alerting claim, and it is evaluated on its own terms below.


## 3. Attribution: what is the model using, and when?

Global importance on this dataset describes monsoon air, because 40% of the rows are
monsoon rows. Two breakdowns fix that: by season, and restricted to extreme rows. If the
new physical features matter anywhere, it is in the second one.

In [6]:
# ============================================================================
# SHAP - global, by season, and on extreme rows only
# ============================================================================
tr, te, _, _ = FOLDS[-1]
y_tr, y_te = y[tr].astype("float64"), y[te].astype("float64")
fit_y = np.log1p(np.clip(y_tr, 0, None)) if USE_LOG else y_tr
w = ev.tail_weights(y_tr) if USE_WEIGHTS else None
final_model = LGBMRegressor(n_estimators=FINAL_TREES, random_state=42, verbose=-1,
                            n_jobs=-1, **BEST_PARAMS)
if QUANTILE is not None:
    final_model.set_params(objective="quantile", alpha=QUANTILE)
final_model.fit(X[tr], fit_y, sample_weight=w)

GROUP_OF = {c: g for g, cols in GROUPS.items() for c in cols}
shap_tables = {}
if HAS_SHAP:
    rng = np.random.default_rng(42)
    # ------------------------------------------------------------------
    # WHICH ROWS TO EXPLAIN
    # ------------------------------------------------------------------
    # Sampling the last fold's test block alone would make the two views the
    # roadmap asks for impossible: that block is entirely monsoon, so "by
    # season" collapses to one season, and monsoon air is clean, so the extreme
    # slice came out at 40 rows. The pool is therefore the union of all fold
    # test blocks, which spans post-monsoon, winter, pre-monsoon and monsoon.
    #
    # Rows before the final model's training cut are in-sample for it. That is
    # acceptable here and only here: SHAP answers "which features does the model
    # use", not "how accurate is it". Every accuracy number in this notebook is
    # still measured out-of-fold.
    # ------------------------------------------------------------------
    pool = np.unique(np.concatenate([f_te for _, f_te, _, _ in FOLDS]))
    sample = pool[rng.choice(len(pool), size=min(20000, len(pool)), replace=False)]
    # A second, deliberately extreme-enriched sample. Keeping it separate leaves
    # the global attribution above representative instead of tilting it toward
    # the tail by construction.
    ext_pool = pool[y[pool] >= 150]
    ext_sample = (ext_pool[rng.choice(len(ext_pool), size=min(5000, len(ext_pool)),
                                      replace=False)]
                  if len(ext_pool) else ext_pool)

    t0 = time.time()
    explainer = shap.TreeExplainer(final_model)
    sv = explainer.shap_values(X[sample])
    sv_ext = explainer.shap_values(X[ext_sample]) if len(ext_sample) else None
    print(f"SHAP over {len(sample):,} representative rows spanning "
          f"{meta.loc[sample, 'season'].nunique()} seasons, plus "
          f"{len(ext_sample):,} extreme rows, in {time.time()-t0:.0f}s")

    def shap_frame(mask, label, values=None):
        vals = np.abs((sv if values is None else values)[mask]).mean(axis=0)
        f = pd.DataFrame({"feature": FEATURES, "mean_abs_shap": vals})
        f["group"] = f.feature.map(GROUP_OF)
        return f.sort_values("mean_abs_shap", ascending=False).assign(slice=label)

    all_mask = np.ones(len(sample), bool)
    shap_tables["all"] = shap_frame(all_mask, "all")
    print("\nTop 15 features overall:")
    display(shap_tables["all"].head(15).round(4).reset_index(drop=True))

    print("Attribution by FEATURE GROUP (share of total |SHAP|):")
    grp = (shap_tables["all"].groupby("group").mean_abs_shap.sum()
           .sort_values(ascending=False))
    grp_share = (grp / grp.sum()).round(4)
    display(pd.DataFrame({"total_abs_shap": grp.round(4), "share": grp_share}))

    # Extreme rows only. This is the slice the whole project is about.
    if sv_ext is not None and len(ext_sample) >= 200:
        ext = np.ones(len(ext_sample), bool)
        shap_tables["extreme"] = shap_frame(ext, "extreme", values=sv_ext)
        print(f"Top 15 features on {len(ext_sample):,} EXTREME rows "
              f"(PM2.5 >= 150) - the slice the roadmap exists to fix:")
        display(shap_tables["extreme"].head(15).round(4).reset_index(drop=True))
        ge = (shap_tables["extreme"].groupby("group").mean_abs_shap.sum()
              .sort_values(ascending=False))
        comparison = pd.DataFrame({
            "share_all": (grp / grp.sum()),
            "share_extreme": (ge / ge.sum()),
        }).fillna(0)
        comparison["shift"] = comparison.share_extreme - comparison.share_all
        print("Group importance: overall vs during extreme air.")
        print("A positive shift means the group matters MORE when it is dangerous, "
              "which is the justification for having acquired it:")
        display(comparison.sort_values("shift", ascending=False).round(4))
    else:
        print(f"Only {len(ext_sample)} extreme rows available across all fold test "
              "blocks - too few for a stable extreme-slice attribution.")

    for season, g in meta.loc[sample].groupby("season", observed=True):
        m = np.isin(sample, g.index.to_numpy())
        if m.sum() >= 500:
            shap_tables[f"season_{season}"] = shap_frame(m, f"season_{season}")
    seasons_present = [k for k in shap_tables if k.startswith("season_")]
    if seasons_present:
        top_by_season = pd.DataFrame({
            k.replace("season_", ""): shap_tables[k].head(8).feature.to_numpy()
            for k in seasons_present})
        print("Top 8 features by season - a stable list means the model uses the "
              "same physics year-round; a shifting one means it switches regimes:")
        display(top_by_season)
else:
    print("shap not installed - skipping attribution. Install with: pip install shap")

SHAP over 20,000 representative rows spanning 4 seasons, plus 5,000 extreme rows, in 2s

Top 15 features overall:


,feature,mean_abs_shap,group,slice
0,pm25,8.1659,temporal,all
1,pm25_roll_mean_168h,3.0635,temporal,all
2,pm25_lag_1h,2.0766,temporal,all
3,pm25_roll_mean_72h,2.0167,temporal,all
4,low_pbl_x_high_pm25,1.6228,interaction,all
5,pm25_lag_72h,1.4182,temporal,all
6,pm25_roll_max_24h,1.4072,temporal,all
7,pm25_roll_p95_24h,1.3883,temporal,all
8,pm25_lag_168h,1.1295,temporal,all
9,pm25_lag_24h,0.9719,temporal,all


Attribution by FEATURE GROUP (share of total |SHAP|):


,total_abs_shap,share
group,,
temporal,25.5118,0.7767
interaction,2.0016,0.0609
satellite,1.4588,0.0444
calendar,1.3402,0.0408
regional,0.7480,0.0228
weather,0.7365,0.0224
static,0.5878,0.0179
fire,0.4627,0.0141


Top 15 features on 5,000 EXTREME rows (PM2.5 >= 150) - the slice the roadmap exists to fix:


,feature,mean_abs_shap,group,slice
0,pm25,33.5386,temporal,extreme
1,pm25_lag_1h,9.5381,temporal,extreme
2,pm25_roll_mean_168h,8.6029,temporal,extreme
3,low_pbl_x_high_pm25,8.1634,interaction,extreme
4,pm25_roll_mean_72h,7.1723,temporal,extreme
5,pm25_roll_p95_24h,5.4554,temporal,extreme
6,pm25_lag_24h,5.4436,temporal,extreme
7,pm25_lag_72h,5.0473,temporal,extreme
8,pm25_lag_168h,4.5220,temporal,extreme
9,pm25_lag_48h,3.9654,temporal,extreme


Group importance: overall vs during extreme air.
A positive shift means the group matters MORE when it is dangerous, which is the justification for having acquired it:


,share_all,share_extreme,shift
group,,,
interaction,0.0609,0.0735,0.0125
calendar,0.0408,0.0466,0.0058
regional,0.0228,0.0269,0.0041
fire,0.0141,0.0152,0.0011
satellite,0.0444,0.0419,-0.0025
temporal,0.7767,0.7736,-0.0030
static,0.0179,0.0103,-0.0076
weather,0.0224,0.0120,-0.0104


Top 8 features by season - a stable list means the model uses the same physics year-round; a shifting one means it switches regimes:


,monsoon,post_monsoon,pre_monsoon,winter
0,pm25,pm25,pm25,pm25
1,pm25_roll_mean_168h,pm25_roll_mean_168h,pm25_roll_mean_168h,pm25_roll_mean_168h
2,pm25_roll_mean_72h,pm25_lag_1h,pm25_lag_1h,pm25_lag_1h
3,pm25_lag_1h,pm25_roll_mean_72h,pm25_roll_mean_72h,pm25_roll_mean_72h
4,low_pbl_x_high_pm25,low_pbl_x_high_pm25,low_pbl_x_high_pm25,low_pbl_x_high_pm25
5,pm25_roll_max_24h,pm25_lag_72h,pm25_roll_max_24h,pm25_lag_72h
6,pm25_roll_p95_24h,pm25_roll_p95_24h,pm25_lag_72h,pm25_roll_p95_24h
7,pm25_lag_72h,pm25_roll_max_24h,pm25_roll_p95_24h,pm25_roll_max_24h


In [7]:
# ============================================================================
# PERMUTATION IMPORTANCE - a check on SHAP, not a duplicate of it
# ============================================================================
# SHAP explains what the model USES. Permutation importance measures what the
# model NEEDS: shuffle a column and see how much accuracy is lost. They diverge
# when features are correlated - SHAP splits credit between collinear features
# while permutation shows neither is individually essential. Both readings are
# useful and neither alone is trustworthy on 184 correlated features.
#
# Run at the GROUP level: permuting one of nine near-identical PM2.5 lags barely
# moves the score because the others carry the same information, which makes a
# per-column permutation on this feature set close to meaningless.
#
# WHICH FOLD. Permutation importance must be measured out-of-sample, because it
# reads accuracy loss. The final model's only unseen block is fold 5 - monsoon,
# clean air, a few hundred extreme rows - and "what the model needs in air nobody
# needs a warning about" is not the question this project asks. So the fold with
# the most extreme test rows is refitted on its own training window and permuted
# on its own test block: still strictly out-of-sample, but during episodes.
# ============================================================================
rng = np.random.default_rng(42)
_ext_per_fold = [int((y[f_te] >= 150).sum()) for _, f_te, _, _ in FOLDS]
PERM_FOLD = int(np.argmax(_ext_per_fold))
p_tr, p_te, _, _ = FOLDS[PERM_FOLD]
print(f"Permuting on fold {PERM_FOLD + 1} "
      f"({meta.loc[p_te, 'season'].mode().iloc[0]}, "
      f"{_ext_per_fold[PERM_FOLD]:,} extreme test rows), refitted on that fold's "
      f"training window so the measurement stays out-of-sample.")

_p_fit_y = (np.log1p(np.clip(y[p_tr].astype("float64"), 0, None)) if USE_LOG
            else y[p_tr].astype("float64"))
perm_model = LGBMRegressor(n_estimators=FINAL_TREES, random_state=42, verbose=-1,
                           n_jobs=-1, **BEST_PARAMS)
if QUANTILE is not None:
    perm_model.set_params(objective="quantile", alpha=QUANTILE)
perm_model.fit(X[p_tr], _p_fit_y,
               sample_weight=ev.tail_weights(y[p_tr]) if USE_WEIGHTS else None)

perm_sample = p_te[rng.choice(len(p_te), size=min(60000, len(p_te)), replace=False)]
X_perm = X[perm_sample].copy()
y_perm = y[perm_sample].astype("float64")


def scored(mat):
    raw = perm_model.predict(mat)
    pred = np.clip(np.expm1(raw) if USE_LOG else raw, 0, None)
    return ev.regression_metrics(y_perm, pred), ev.extreme_bias(y_perm, pred)


ref_metrics, ref_bias = scored(X_perm)
print(f"Reference on {len(perm_sample):,} rows "
      f"({int((y_perm >= 150).sum()):,} extreme): RMSE {ref_metrics['RMSE']:.3f}, "
      f"R2 {ref_metrics['R2']:.4f}, extreme bias {ref_bias:+.2f}\n")

perm_rows = []
for group, cols in GROUPS.items():
    idx = [FEATURES.index(c) for c in cols]
    shuffled = X_perm.copy()
    order = rng.permutation(len(shuffled))
    shuffled[:, idx] = shuffled[order][:, idx]     # permute the block together
    m, b = scored(shuffled)
    perm_rows.append({"group": group, "n_features": len(cols),
                      "RMSE_increase": m["RMSE"] - ref_metrics["RMSE"],
                      "R2_drop": ref_metrics["R2"] - m["R2"],
                      "extreme_bias_change": b - ref_bias})
    del shuffled

del perm_model, X_perm
gc.collect()
perm = pd.DataFrame(perm_rows).sort_values("RMSE_increase", ascending=False)
display(perm.round(4).reset_index(drop=True))
print("RMSE_increase is what the model loses without the group. A group with a "
      "near-zero value is either genuinely uninformative or fully substitutable "
      "by another group - permutation cannot distinguish those two, which is why "
      "the ablation in notebook 06 (which removes groups at TRAINING time) is the "
      "stronger evidence.")

Permuting on fold 1 (post_monsoon, 6,018 extreme test rows), refitted on that fold's training window so the measurement stays out-of-sample.


Reference on 60,000 rows (1,813 extreme): RMSE 33.415, R2 0.4495, extreme bias -84.81



,group,n_features,RMSE_increase,R2_drop,extreme_bias_change
0,temporal,42,20.7944,0.8983,-63.6498
1,satellite,22,0.4339,0.0144,0.7548
2,interaction,6,0.1711,0.0057,-1.9441
3,fire,46,0.1243,0.0041,-0.2301
4,weather,28,0.1051,0.0035,1.1884
5,regional,15,0.0844,0.0028,-0.1311
6,calendar,18,0.0298,0.0010,0.0235
7,static,7,-0.0531,-0.0017,1.1169


RMSE_increase is what the model loses without the group. A group with a near-zero value is either genuinely uninformative or fully substitutable by another group - permutation cannot distinguish those two, which is why the ablation in notebook 06 (which removes groups at TRAINING time) is the stronger evidence.


## 4. The acceptance gate

Roadmap §23. Measured, reported, and *not* negotiated.

Two properties of the gate are worth stating plainly before the result, because both affect
how the numbers should be read — neither is a reason to move a threshold.

**The lead-time gate is capped by construction.** The threshold is 24 hours and the hazard
target looks exactly 24 hours ahead, so lead time cannot exceed 24 hours. Passing therefore
requires detecting essentially every episode at the first possible opportunity.

**The R² gate is horizon-sensitive, and this is measured at t+24h.** Phase 5 reported a
rolling-CV mean R² of 0.67 at **t+1h**; the same family of models scores far lower at
t+24h simply because 24-hour-ahead prediction is a harder problem, not because anything
regressed. The roadmap does not name a horizon for the 0.75 figure. It is applied here at
the operational horizon of t+24h, which is the strict reading, and the horizon is recorded
alongside the result so the comparison against Phase 5 stays honest in both directions.

In [8]:
# ============================================================================
# SECTION-23 ACCEPTANCE GATE
# ============================================================================
# AGENTS.md: "The promotion gate is load-bearing. Do not relax a threshold to
# make a model pass; fix the model or leave it at VALIDATION."
# ============================================================================
best_model_row = models_only.iloc[0]
measured = {
    "rolling_cv_mean_r2": float(best_model_row.mean_R2),
    "extreme_recall": float(best_model_row.extreme_recall),
    "extreme_precision": float(best_model_row.extreme_precision),
    "median_lead_time_h": float(
        (haz or {}).get("event_lead_time", {}).get("median_lead_time_h", np.nan)),
}
# The alerting gates belong to the hazard CLASSIFIER, not to the regression -
# that separation is the whole argument of notebook 07. Where the classifier
# reported a number, it is the one that counts.
if haz:
    sel = haz["hazard"]["selected_variant"]
    row = next(r for r in haz["hazard"]["calibration_comparison"]
               if r["variant"] == sel)
    measured["extreme_recall"] = float(row["recall"])
    measured["extreme_precision"] = float(row["precision"])
    source = f"hazard classifier ({sel}) from notebook 07"
else:
    source = "regression thresholding (notebook 07 has not run)"

print(f"Alerting metrics sourced from: {source}")
print(f"Regression metrics measured at: t+{PRIMARY_H}h over {len(FOLDS)} "
      f"rolling-origin folds covering {sorted(set(fold_seasons))}")
print("Phase 5's comparable R2 of 0.67 was measured at t+1h, a materially easier "
      "problem; do not read the two side by side without the horizon.\n")
gate_rows = []
for name, threshold in ACCEPTANCE_GATES.items():
    got = measured[name]
    passed = bool(not np.isnan(got) and got >= threshold)
    gate_rows.append({"gate": name, "threshold": threshold,
                      "measured": got, "pass": passed})
    flag = "PASS" if passed else ("NO DATA" if np.isnan(got) else "FAIL")
    print(f"  [{flag:7}] {name:24} required >= {threshold:<6} measured "
          f"{got if not np.isnan(got) else float('nan'):.4f}")

gates = pd.DataFrame(gate_rows)
all_passed = bool(gates["pass"].all())
STATUS = "PRODUCTION_CANDIDATE" if all_passed else "VALIDATION"

print(f"\nGates passed: {int(gates['pass'].sum())}/{len(gates)}")
print(f"STATUS: {STATUS}")
if not all_passed:
    failed = gates[~gates["pass"]]
    print("\nNot promoted. Failing gates:")
    for r in failed.itertuples():
        print(f"  - {r.gate}: {r.measured:.4f} < {r.threshold}")
    print("\nThe thresholds are NOT being relaxed. What would actually move them, "
          "in order of expected effect:")
    print("  1. More post-monsoon cycles. The dataset holds ONE burning season, so "
          "     the model has seen the phenomenon it must predict exactly once. This "
          "     is the roadmap's deferred Sprint 1 (CPCB 3-5 year rebuild).")
    print("  2. Forecast weather at t+h instead of observations at t (roadmap §30). "
          "     A 24h forecast currently uses 24-hour-old meteorology.")
    print("  3. Population, road and land-use density (roadmap §15), which are the "
          "     acquisition gaps recorded in the Phase-7 driver manifest.")
    print("  4. LSTM / sequence models (roadmap §24-34), explicitly out of scope "
          "     for this pass and correctly sequenced after the above.")

Alerting metrics sourced from: hazard classifier (platt) from notebook 07
Regression metrics measured at: t+24h over 5 rolling-origin folds covering ['monsoon', 'post_monsoon', 'pre_monsoon', 'winter']
Phase 5's comparable R2 of 0.67 was measured at t+1h, a materially easier problem; do not read the two side by side without the horizon.

  [FAIL   ] rolling_cv_mean_r2       required >= 0.75   measured 0.2261
  [FAIL   ] extreme_recall           required >= 0.7    measured 0.6746
  [PASS   ] extreme_precision        required >= 0.25   measured 0.3489
  [PASS   ] median_lead_time_h       required >= 24.0   measured 24.0000

Gates passed: 2/4
STATUS: VALIDATION

Not promoted. Failing gates:
  - rolling_cv_mean_r2: 0.2261 < 0.75
  - extreme_recall: 0.6746 < 0.7

The thresholds are NOT being relaxed. What would actually move them, in order of expected effect:
  1. More post-monsoon cycles. The dataset holds ONE burning season, so      the model has seen the phenomenon it must predict exactl

In [9]:
# ============================================================================
# PERSIST TUNING ARTIFACTS AND THE GATE DECISION
# ============================================================================
trials.to_csv(OUT_DIR / "optuna_trials.csv", index=False)
tuning_effect.to_csv(OUT_DIR / "tuned_vs_untuned.csv", index=False)
final.drop(columns=["abs_extreme_bias"]).to_csv(OUT_DIR / "final_comparison.csv",
                                                index=False)
perm.to_csv(OUT_DIR / "permutation_importance_by_group.csv", index=False)
gates.to_csv(OUT_DIR / "acceptance_gates.csv", index=False)
tuned_folds.to_csv(OUT_DIR / "tuned_per_fold.csv", index=False)
for name, table in shap_tables.items():
    table.to_csv(OUT_DIR / f"shap_{name}.csv", index=False)

# The tuned regressor is saved on its own. It is deliberately NOT merged into the
# notebook-07 bundle: that bundle passed a round-trip check as evaluated, and
# swapping a model into it here would invalidate those measurements.
TUNED_FILE = OUT_DIR / "pm25_tuned_regressor.joblib"
joblib.dump({
    "schema_version": "1.0.0",
    "dataset_version": manifest["dataset_version"],
    "dataset_fingerprint": manifest["sha256_16"],
    "created_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "feature_names": FEATURES,
    "target": TARGET,
    "params": BEST_PARAMS,
    "n_estimators": FINAL_TREES,
    "loss_config": {"log1p": USE_LOG, "tail_weights": USE_WEIGHTS,
                    "quantile": QUANTILE, "selected_by": CHOSEN},
    "model": final_model,
    "status": STATUS,
}, TUNED_FILE, compress=3)

report = {
    "generated_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "dataset": {"version": manifest["dataset_version"],
                "fingerprint": manifest["sha256_16"],
                "rows": manifest["rows"], "n_features": manifest["n_features"]},
    "protocol": {
        "cv": "purged rolling-origin (expanding window)",
        "n_splits": len(FOLDS),
        "horizon_hours": PRIMARY_H,
        "test_fold_seasons": fold_seasons,
        "horizon_caveat": "Phase 5 reported rolling-CV mean R2 0.67 at t+1h. These "
                          f"metrics are at t+{PRIMARY_H}h, a materially harder "
                          "problem; the two are not comparable without the horizon.",
        "purge_hours": PRIMARY_H, "embargo_hours": EMBARGO_H,
        "random_kfold": "FORBIDDEN - autoregressive features leak across a shuffle",
        "optuna_objective": "mean(RMSE) + std(RMSE) across folds, i.e. tuned for "
                            "fold-to-fold stability rather than a best-season score",
        "optuna_constraint": "a trial may not worsen |extreme_bias| against the "
                             "untuned configuration",
        "search_trees": SEARCH_TREES, "final_trees": FINAL_TREES,
        "n_trials": len(study.trials),
    },
    "loss_config_from_notebook_06": {"experiment": CHOSEN, "log1p": USE_LOG,
                                     "tail_weights": USE_WEIGHTS,
                                     "quantile": QUANTILE},
    "untuned_reference": {"params": BASE_PARAMS, "objective": BASELINE_OBJ,
                          "extreme_bias": BASELINE_XBIAS},
    "best_params": BEST_PARAMS,
    "selected_from": tuned_from,
    "eligible_trials": int(len(eligible)),
    "total_trials": int(len(trials)),
    "tuning_effect": tuning_effect.to_dict("records"),
    "tuning_gain_within_fold_spread": bool(abs(d_rmse) < spread),
    "final_comparison": final.drop(columns=["abs_extreme_bias"]).to_dict("records"),
    "permutation_importance_by_group": perm.to_dict("records"),
    "shap_available": HAS_SHAP,
    "acceptance_gates": gates.to_dict("records"),
    "all_gates_passed": all_passed,
    "status": STATUS,
    "alerting_metrics_source": source,
    "artifact": str(TUNED_FILE.relative_to(PROJECT_ROOT)),
    "blockers_if_not_promoted": [] if all_passed else [
        "one post-monsoon cycle in the dataset (roadmap Sprint 1, deferred)",
        "weather observations at t rather than forecasts valid at t+h (roadmap 30)",
        "population/road/land-use density not acquired (roadmap 15)",
        "sequence models not attempted (roadmap 24-34, out of scope this pass)",
    ],
}
with open(OUT_DIR / "tuning_report.json", "w") as f:
    json.dump(report, f, indent=2, default=str)

print("PHASE 7 COMPLETE\n")
print(f"  optuna trials      : {len(study.trials)} ({len(eligible)} eligible)")
print(f"  selected           : {tuned_from}")
print(f"  tuned mean R2      : {tuning_effect.mean_R2.iloc[1]:+.4f} "
      f"(untuned {tuning_effect.mean_R2.iloc[0]:+.4f})")
print(f"  tuned extreme bias : {tuning_effect.extreme_bias.iloc[1]:+.2f} ug/m3 "
      f"(untuned {tuning_effect.extreme_bias.iloc[0]:+.2f})")
print(f"  best model overall : {models_only.iloc[0].model}")
print(f"  gates              : {int(gates['pass'].sum())}/{len(gates)} passed")
print(f"  STATUS             : {STATUS}")
print(f"  artifact           : {TUNED_FILE.name} "
      f"({TUNED_FILE.stat().st_size/1e6:.2f} MB)")
print(f"\nWrote to {OUT_DIR}:")
for p in sorted(OUT_DIR.glob("*")):
    print(f"  {p.name:42} {p.stat().st_size/1e3:9.1f} KB")

PHASE 7 COMPLETE

  optuna trials      : 20 (19 eligible)
  selected           : optuna trial 18
  tuned mean R2      : +0.2261 (untuned +0.2346)
  tuned extreme bias : -130.62 ug/m3 (untuned -135.34)
  best model overall : lightgbm_tuned
  gates              : 2/4 passed
  STATUS             : VALIDATION
  artifact           : pm25_tuned_regressor.joblib (0.36 MB)

Wrote to /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator/artifacts/pm25/phase7/tuning:
  acceptance_gates.csv                             0.2 KB
  final_comparison.csv                             0.9 KB
  optuna_trials.csv                                5.2 KB
  permutation_importance_by_group.csv              0.6 KB
  pm25_tuned_regressor.joblib                    355.7 KB
  shap_all.csv                                     9.0 KB
  shap_extreme.csv                                 9.6 KB
  shap_season_monsoon.csv                         11.0 KB
  shap_season_post_monsoon.csv       